# 01 — Setup Environment & Download Dataset INDOMEME

**Proyek:** IndoMeme-XAI — Hate Speech Detection pada Meme Indonesia  
**Tujuan notebook ini:**
1. Install semua dependencies yang diperlukan
2. Download dataset INDOMEME dari HuggingFace (`aiatums/INDOMEME`)
3. Normalisasi nama split → selalu `train` / `validation` / `test`
4. **Simpan raw parquet** ke `data/raw/`
5. Simpan gambar ke `data/images/<split>/`
6. Export CSV per split ke `data/processed/`
7. Verifikasi integritas dataset

---
> **Catatan:** Notebook ini dijalankan **sekali** untuk setup awal. Setelah dataset tersimpan lokal, notebook EDA dan training tidak perlu re-download.

## 0. Struktur Folder Project

```
INDOMEME-XAI/
├── data/
│   ├── raw/                  ← dataset mentah dari HuggingFace (parquet)
│   │   ├── train.parquet
│   │   ├── validation.parquet
│   │   └── test.parquet
│   ├── processed/            ← CSV bersih siap EDA
│   │   ├── train.csv
│   │   ├── validation.csv
│   │   ├── test.csv
│   │   └── all_splits.csv
│   ├── images/               ← gambar meme (dari kolom 'image')
│   │   ├── train/
│   │   ├── validation/
│   │   └── test/
│   └── dataset_metadata.json
├── notebooks/
│   ├── 01_setup_download.ipynb   ← notebook ini
│   ├── 02_EDA_mendalam.ipynb
│   └── 03_preprocessing.ipynb    (nanti)
├── src/
└── outputs/
    ├── checkpoints/
    ├── figures/
    └── results/
```

## 1. Buat Struktur Folder

In [1]:
import os
from pathlib import Path

# ============================================================
# SESUAIKAN: path root project Anda
# Jika notebook ada di notebooks/ → .parent naik satu level
# Jika notebook ada di root project → gunakan Path('.').resolve()
# ============================================================
PROJECT_ROOT = Path(".").resolve().parent  # asumsi notebook ada di notebooks/
print(f"Project root: {PROJECT_ROOT}")

folders = [
    "data/raw",
    "data/processed",
    "data/images/train",
    "data/images/validation",
    "data/images/test",
    "notebooks",
    "src",
    "outputs/checkpoints",
    "outputs/figures",
    "outputs/results",
]

for folder in folders:
    path = PROJECT_ROOT / folder
    path.mkdir(parents=True, exist_ok=True)
    print(f"  ✓ {path}")

RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
IMAGES_ROOT   = PROJECT_ROOT / "data" / "images"

print("\n✅ Semua folder berhasil dibuat.")

Project root: D:\PROYEK ML DAN AI\IndoMeme-XAI
  ✓ D:\PROYEK ML DAN AI\IndoMeme-XAI\data\raw
  ✓ D:\PROYEK ML DAN AI\IndoMeme-XAI\data\processed
  ✓ D:\PROYEK ML DAN AI\IndoMeme-XAI\data\images\train
  ✓ D:\PROYEK ML DAN AI\IndoMeme-XAI\data\images\validation
  ✓ D:\PROYEK ML DAN AI\IndoMeme-XAI\data\images\test
  ✓ D:\PROYEK ML DAN AI\IndoMeme-XAI\notebooks
  ✓ D:\PROYEK ML DAN AI\IndoMeme-XAI\src
  ✓ D:\PROYEK ML DAN AI\IndoMeme-XAI\outputs\checkpoints
  ✓ D:\PROYEK ML DAN AI\IndoMeme-XAI\outputs\figures
  ✓ D:\PROYEK ML DAN AI\IndoMeme-XAI\outputs\results

✅ Semua folder berhasil dibuat.


## 2. Install Dependencies

In [2]:
!pip install -q datasets huggingface_hub
!pip install -q pandas numpy matplotlib seaborn
!pip install -q Pillow tqdm
!pip install -q pyarrow fastparquet
!pip install -q wordcloud scikit-learn scipy

print("\n✅ Semua dependencies berhasil diinstall.")


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip



✅ Semua dependencies berhasil diinstall.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Import Libraries

In [3]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
from datasets import load_dataset, DatasetDict

print("✅ Import berhasil.")
print(f"  pandas : {pd.__version__}")
print(f"  numpy  : {np.__version__}")

d:\PROYEK ML DAN AI\IndoMeme-XAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Import berhasil.
  pandas : 3.0.0
  numpy  : 1.26.4


## 4. Download Dataset dari HuggingFace

In [4]:
print("📥 Mendownload dataset INDOMEME dari HuggingFace...")
print("   (Ini mungkin memakan waktu beberapa menit)\n")

dataset = load_dataset(
    "aiatums/INDOMEME",
    trust_remote_code=True,
)

print("\n✅ Dataset berhasil didownload!")
print(f"Split yang tersedia: {list(dataset.keys())}")
for split_name, split_data in dataset.items():
    print(f"  {split_name:15s}: {len(split_data):,} sampel")

📥 Mendownload dataset INDOMEME dari HuggingFace...
   (Ini mungkin memakan waktu beberapa menit)


✅ Dataset berhasil didownload!
Split yang tersedia: ['train', 'test']
  train          : 4,018 sampel
  test           : 1,005 sampel


In [5]:
# -------------------------------------------------------
# Normalisasi nama split → selalu train / validation / test
# HuggingFace kadang pakai 'val', 'dev', atau nama lain
# -------------------------------------------------------
def normalize_split_name(key):
    k = key.lower()
    if k == 'train':
        return 'train'
    elif k in ('validation', 'valid', 'val', 'dev'):
        return 'validation'
    elif k in ('test', 'eval'):
        return 'test'
    return k  # simpan apa adanya jika tidak dikenal

# Buat dataset baru dengan nama split yang sudah dinormalisasi
normalized_dataset = DatasetDict({
    normalize_split_name(k): v for k, v in dataset.items()
})

print("Nama split setelah normalisasi:")
for split_name, split_data in normalized_dataset.items():
    print(f"  {split_name:15s}: {len(split_data):,} sampel")

# -------------------------------------------------------
# Jika 'validation' tidak ada → buat dari 15% data train
# -------------------------------------------------------
if 'validation' not in normalized_dataset:
    print("\n⚠️  Split 'validation' tidak ditemukan!")
    print("   → Membuat validation set dari 15% data train (seed=42)...")
    
    train_val_split = normalized_dataset['train'].train_test_split(
        test_size=0.15, seed=42
    )
    normalized_dataset = DatasetDict({
        'train':      train_val_split['train'],
        'validation': train_val_split['test'],
        **{k: v for k, v in normalized_dataset.items() if k != 'train'}
    })
    
    print(f"   ✅ Train baru  : {len(normalized_dataset['train']):,} sampel")
    print(f"   ✅ Validation  : {len(normalized_dataset['validation']):,} sampel")
else:
    print("\n✅ Split 'validation' ditemukan — tidak perlu dibuat dari train.")

# Gunakan normalized_dataset untuk semua langkah selanjutnya
dataset = normalized_dataset
print(f"\nFinal splits: {list(dataset.keys())}")

Nama split setelah normalisasi:
  train          : 4,018 sampel
  test           : 1,005 sampel

⚠️  Split 'validation' tidak ditemukan!
   → Membuat validation set dari 15% data train (seed=42)...
   ✅ Train baru  : 3,415 sampel
   ✅ Validation  : 603 sampel

Final splits: ['train', 'validation', 'test']


In [6]:
# Tampilkan info kolom
print("📋 Informasi fitur dataset:")
print(dataset['train'].features)
print("\n📋 Contoh satu baris (tanpa kolom image):")
sample = {k: v for k, v in dataset['train'][0].items() if k != 'image'}
for key, val in sample.items():
    display_val = str(val)
    if len(display_val) > 120:
        display_val = display_val[:120] + '...'
    print(f"  {key:20s}: {display_val}")

📋 Informasi fitur dataset:
{'image': Image(decode=True, id=None), 'image_path': Value(dtype='string', id=None), 'hate_final': Value(dtype='string', id=None), 'appropriate_final': Value(dtype='string', id=None), 'topic': Value(dtype='string', id=None), 'topic_id': Value(dtype='string', id=None), 'ocr': Value(dtype='string', id=None), 'caption': Value(dtype='string', id=None)}

📋 Contoh satu baris (tanpa kolom image):
  image_path          : 956949229800835.jpeg
  hate_final          : not hate
  appropriate_final   : appropriate
  topic               : social sub-groups
  topic_id            : Kelompok Sosial
  ocr                 : ketika ada rumah tangga yang lagi berantem
tingga tetangga yang nguping :
  caption             : Gambar ini secara humoris menggambarkan fenomena tetangga yang menguping perselisihan rumah tangga, dilambangkan oleh ku...


## 5. Simpan Raw Data (Parquet)

Menyimpan data mentah ke `data/raw/` agar bisa di-load ulang tanpa koneksi internet.

In [7]:
# Simpan setiap split sebagai parquet (kolom image dikecualikan untuk efisiensi)
# Gambar akan disimpan terpisah di langkah berikutnya

print("💾 Menyimpan raw data ke data/raw/ ...")

for split_name, split_data in dataset.items():
    out_path = RAW_DIR / f"{split_name}.parquet"
    
    # Konversi ke DataFrame, drop kolom 'image' (PIL object, tidak bisa disimpan di parquet biasa)
    rows = []
    for row in tqdm(split_data, desc=f"Konversi raw [{split_name}]", leave=False):
        record = {k: v for k, v in row.items() if k != 'image'}
        rows.append(record)
    
    df_raw = pd.DataFrame(rows)
    df_raw.to_parquet(out_path, index=False)
    
    size_mb = out_path.stat().st_size / (1024 * 1024)
    print(f"  ✅ {split_name:15s}: {len(df_raw):,} baris → {out_path.name} ({size_mb:.1f} MB)")

print("\n✅ Semua raw data tersimpan di data/raw/")
print("   (Kolom 'image' tidak disimpan di parquet — gambar ada di data/images/)")

💾 Menyimpan raw data ke data/raw/ ...


  ✅ train          : 3,415 baris → train.parquet (0.6 MB)


  ✅ validation     : 603 baris → validation.parquet (0.1 MB)


  ✅ test           : 1,005 baris → test.parquet (0.2 MB)

✅ Semua raw data tersimpan di data/raw/
   (Kolom 'image' tidak disimpan di parquet — gambar ada di data/images/)


## 6. Simpan Gambar ke Disk

In [8]:
def save_images_from_split(split_data, split_name, images_root):
    """
    Simpan gambar dari HuggingFace dataset ke folder lokal.
    Mengembalikan (saved_paths, failed_list).
    """
    save_dir = Path(images_root) / split_name
    save_dir.mkdir(parents=True, exist_ok=True)
    
    saved_paths = []
    failed = []
    
    for i, row in enumerate(tqdm(split_data, desc=f"Simpan gambar [{split_name}]")):
        try:
            img = row['image']  # PIL Image object
            
            # Nama file: gunakan image_path dari dataset, fallback ke index
            img_name = str(row.get('image_path') or f"meme_{i:05d}.jpg").strip()
            if not Path(img_name).suffix:
                img_name += ".jpg"
            # Amankan nama file (hapus path separator)
            img_name = Path(img_name).name
            
            save_path = save_dir / img_name
            
            # Konversi ke RGB jika perlu
            if img.mode not in ('RGB', 'L'):
                img = img.convert('RGB')
            
            img.save(save_path, quality=95)
            saved_paths.append(str(save_path))
            
        except Exception as e:
            failed.append((i, str(e)))
            saved_paths.append('')
    
    print(f"  ✅ {split_name}: {len(saved_paths) - len(failed):,} gambar tersimpan"
          f"{f', {len(failed)} gagal' if failed else ''}")
    if failed:
        for idx, err in failed[:3]:
            print(f"     index {idx}: {err}")
    
    return saved_paths, failed

In [9]:
image_paths_per_split = {}
failed_per_split = {}

for split_name in dataset.keys():
    paths, fails = save_images_from_split(
        dataset[split_name], split_name, IMAGES_ROOT
    )
    image_paths_per_split[split_name] = paths
    failed_per_split[split_name] = fails

print("\n✅ Semua gambar berhasil disimpan!")

Simpan gambar [train]: 100%|██████████| 3415/3415 [00:36<00:00, 92.37it/s] 


  ✅ train: 3,415 gambar tersimpan


Simpan gambar [validation]: 100%|██████████| 603/603 [00:06<00:00, 88.73it/s] 


  ✅ validation: 603 gambar tersimpan


Simpan gambar [test]: 100%|██████████| 1005/1005 [00:12<00:00, 83.44it/s]

  ✅ test: 1,005 gambar tersimpan

✅ Semua gambar berhasil disimpan!


## 7. Buat DataFrame dan Simpan CSV

In [10]:
def dataset_split_to_df(split_data, split_name, image_paths):
    """
    Konversi HuggingFace dataset split ke pandas DataFrame.
    Kolom 'image' (PIL object) diganti dengan path file lokal.
    """
    rows = []
    for i, row in enumerate(tqdm(split_data, desc=f"DataFrame [{split_name}]", leave=False)):
        record = {
            'idx':               i,
            'split':             split_name,
            'image_path':        row.get('image_path', f'meme_{i:05d}.jpg'),
            'local_image_path':  image_paths[i] if i < len(image_paths) else '',
            'hate_final':        row.get('hate_final', ''),
            'appropriate_final': row.get('appropriate_final', ''),
            'topic':             row.get('topic', ''),
            'topic_id':          row.get('topic_id', ''),
            'ocr':               row.get('ocr', ''),
            'caption':           row.get('caption', ''),
        }
        rows.append(record)
    return pd.DataFrame(rows)


dfs = {}
for split_name in dataset.keys():
    df = dataset_split_to_df(
        dataset[split_name], split_name, image_paths_per_split[split_name]
    )
    dfs[split_name] = df
    print(f"  {split_name:15s}: shape = {df.shape}")

print("\n✅ DataFrame berhasil dibuat!")

  train          : shape = (3415, 10)


  validation     : shape = (603, 10)


  test           : shape = (1005, 10)

✅ DataFrame berhasil dibuat!


In [11]:
# Preview
print("Preview split 'train':")
dfs['train'][['image_path', 'hate_final', 'appropriate_final', 'topic', 'ocr', 'caption']].head(3)

Preview split 'train':


,image_path,hate_final,appropriate_final,topic,ocr,caption
0,956949229800835.jpeg,not hate,appropriate,social sub-groups,ketika ada rumah tangga yang lagi berantem\nti...,Gambar ini secara humoris menggambarkan fenome...
1,1138730521627796.jpeg,not hate,appropriate,social sub-groups,List gimmick Madrid hari ini :\n1. Nutup atap ...,Konten visual ini menyajikan daftar humoris be...
2,840053618162156.jpeg,not hate,appropriate,religion,disaat malam ganjil tiba\nakan aku spam ibadah,Visual ini menampilkan karakter animasi yang m...


In [12]:
# Simpan CSV per split
print("💾 Menyimpan CSV ke data/processed/ ...")

# Pastikan semua split tersimpan: train, validation, test
required_splits = ['train', 'validation', 'test']
for split_name in required_splits:
    if split_name not in dfs:
        print(f"  ⚠️  Split '{split_name}' tidak ada — dilewati.")
        continue
    out_path = PROCESSED_DIR / f"{split_name}.csv"
    dfs[split_name].to_csv(out_path, index=False, encoding='utf-8')
    size_kb = out_path.stat().st_size / 1024
    print(f"  ✅ {split_name:15s}: {len(dfs[split_name]):,} baris → {out_path.name} ({size_kb:.0f} KB)")

# Simpan juga split lain yang mungkin ada
for split_name, df in dfs.items():
    if split_name not in required_splits:
        out_path = PROCESSED_DIR / f"{split_name}.csv"
        df.to_csv(out_path, index=False, encoding='utf-8')
        print(f"  ✅ {split_name:15s}: {len(df):,} baris → {out_path.name}")

# Simpan gabungan semua split
df_all = pd.concat(dfs.values(), ignore_index=True)
all_path = PROCESSED_DIR / "all_splits.csv"
df_all.to_csv(all_path, index=False, encoding='utf-8')
print(f"\n  ✅ all_splits.csv  : {len(df_all):,} baris total")

print("\n✅ Semua CSV tersimpan!")

💾 Menyimpan CSV ke data/processed/ ...
  ✅ train          : 3,415 baris → train.csv (1323 KB)
  ✅ validation     : 603 baris → validation.csv (237 KB)
  ✅ test           : 1,005 baris → test.csv (386 KB)

  ✅ all_splits.csv  : 5,023 baris total

✅ Semua CSV tersimpan!


## 8. Verifikasi Akhir — Cek Semua File Tersimpan

In [13]:
print("=" * 60)
print("VERIFIKASI AKHIR")
print("=" * 60)

# 1. Cek file parquet di raw/
print("\n[ data/raw/ ]")
for f in sorted(RAW_DIR.glob("*.parquet")):
    df_check = pd.read_parquet(f)
    print(f"  ✅ {f.name:25s}: {len(df_check):,} baris, {len(df_check.columns)} kolom")

# 2. Cek file CSV di processed/
print("\n[ data/processed/ ]")
for f in sorted(PROCESSED_DIR.glob("*.csv")):
    df_check = pd.read_csv(f)
    print(f"  ✅ {f.name:25s}: {len(df_check):,} baris, {len(df_check.columns)} kolom")

# 3. Cek folder gambar
print("\n[ data/images/ ]")
for split_dir in sorted(IMAGES_ROOT.iterdir()):
    if split_dir.is_dir():
        n_imgs = len(list(split_dir.glob("*")))
        print(f"  ✅ {split_dir.name:15s}: {n_imgs:,} file gambar")

# 4. Test buka gambar pertama dari setiap split
print("\n[ Test buka gambar ]")
for split_name, df in dfs.items():
    valid_paths = df[df['local_image_path'] != '']['local_image_path']
    if len(valid_paths) > 0:
        try:
            img = Image.open(valid_paths.iloc[0])
            print(f"  ✅ {split_name:15s}: size={img.size}, mode={img.mode}")
        except Exception as e:
            print(f"  ❌ {split_name}: {e}")
    else:
        print(f"  ⚠️  {split_name}: tidak ada path gambar valid")

# 5. Verifikasi distribusi label
print("\n[ Distribusi Label ]")
print(f"{'Split':15} | {'N':>6} | {'hate':>8} | {'not hate':>10} | {'hate %':>7}")
print("-" * 55)
for split_name in ['train', 'validation', 'test']:
    if split_name in dfs:
        df = dfs[split_name]
        n = len(df)
        n_hate    = (df['hate_final'] == 'hate').sum()
        n_nothate = (df['hate_final'] != 'hate').sum()
        pct_hate  = n_hate / n * 100
        print(f"{split_name:15} | {n:>6,} | {n_hate:>8,} | {n_nothate:>10,} | {pct_hate:>6.1f}%")
    else:
        print(f"{split_name:15} | ⚠️ tidak ada")

print("\n" + "=" * 60)
print("✅ Setup selesai! Lanjut ke 02_EDA_mendalam.ipynb")
print("=" * 60)

VERIFIKASI AKHIR

[ data/raw/ ]
  ✅ test.parquet             : 1,005 baris, 7 kolom
  ✅ train.parquet            : 3,415 baris, 7 kolom
  ✅ validation.parquet       : 603 baris, 7 kolom

[ data/processed/ ]
  ✅ all_splits.csv           : 5,023 baris, 10 kolom
  ✅ test.csv                 : 1,005 baris, 10 kolom
  ✅ train.csv                : 3,415 baris, 10 kolom
  ✅ validation.csv           : 603 baris, 10 kolom

[ data/images/ ]
  ✅ test           : 1,005 file gambar
  ✅ train          : 3,415 file gambar
  ✅ validation     : 603 file gambar

[ Test buka gambar ]
  ✅ train          : size=(1440, 1800), mode=RGB
  ✅ validation     : size=(828, 680), mode=RGB
  ✅ test           : size=(1080, 1440), mode=RGB

[ Distribusi Label ]
Split           |      N |     hate |   not hate |  hate %
-------------------------------------------------------
train           |  3,415 |    1,173 |      2,242 |   34.3%
validation      |    603 |      213 |        390 |   35.3%
test            |  1,005 |  

## 9. Simpan Metadata Setup

In [14]:
import json
from datetime import datetime

metadata = {
    "dataset": "aiatums/INDOMEME",
    "download_date": datetime.now().isoformat(),
    "splits": {},
    "columns": list(dfs[list(dfs.keys())[0]].columns),
    "label_columns": ["hate_final", "appropriate_final"],
    "task": "hate_speech_detection",
    "primary_label": "hate_final",
    "label_map": {"not hate": 0, "hate": 1},
}

for split_name, df in dfs.items():
    metadata["splits"][split_name] = {
        "n_samples": len(df),
        "hate_dist": df['hate_final'].value_counts().to_dict(),
        "appropriate_dist": df['appropriate_final'].value_counts().to_dict(),
        "csv_path": str(PROCESSED_DIR / f"{split_name}.csv"),
        "parquet_path": str(RAW_DIR / f"{split_name}.parquet"),
        "images_dir": str(IMAGES_ROOT / split_name),
        "n_images_saved": len([p for p in image_paths_per_split[split_name] if p]),
    }

meta_path = PROJECT_ROOT / "data" / "dataset_metadata.json"
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✅ Metadata tersimpan: {meta_path}")
print("\nRingkasan splits:")
for split_name, info in metadata['splits'].items():
    print(f"  {split_name:15s}: {info['n_samples']:,} sampel, {info['n_images_saved']:,} gambar")

✅ Metadata tersimpan: D:\PROYEK ML DAN AI\IndoMeme-XAI\data\dataset_metadata.json

Ringkasan splits:
  train          : 3,415 sampel, 3,415 gambar
  validation     : 603 sampel, 603 gambar
  test           : 1,005 sampel, 1,005 gambar


## Ringkasan File yang Tersimpan

```
data/
├── raw/
│   ├── train.parquet          ← data mentah HuggingFace (tanpa kolom image)
│   ├── validation.parquet
│   └── test.parquet
├── processed/
│   ├── train.csv              ← siap untuk EDA & training
│   ├── validation.csv
│   ├── test.csv
│   └── all_splits.csv         ← gabungan semua split
├── images/
│   ├── train/                 ← gambar training
│   ├── validation/            ← gambar validasi
│   └── test/                  ← gambar test
└── dataset_metadata.json      ← statistik dasar + paths
```

**Next step:** Buka `02_EDA_mendalam.ipynb`